# 09 — Parametric Distributions

![Level](https://img.shields.io/badge/level-intermediate-yellow)
![Python](https://img.shields.io/badge/python-3.10%2B-blue)
![Twiga](https://img.shields.io/badge/twiga-forecast-orange)
![Time](https://img.shields.io/badge/time-~25%20min-lightgrey)

---

**What you'll build**

Probabilistic neural network forecasters using parametric distribution heads (Normal, Laplace, Gamma) trained with negative log-likelihood loss, evaluated on CRPS and calibration diagrams.

**Prerequisites**
- [07 — Neural Networks](07-neural-networks.ipynb) (MLPF, NHiTS, Lightning training loop)
- [08 — Quantile Regression](08-quantile-regression.ipynb) (interval metrics: PICP, NMPI, Winkler)
- Python: basic probability distributions helpful

**Learning objectives**

By the end of this notebook you will be able to:

1. Choose a distribution family (Normal, Laplace, Gamma, LogNormal, Beta) based on signal characteristics
2. Configure and train probabilistic neural networks using negative log-likelihood loss
3. Evaluate probabilistic forecasts with CRPS and Winkler score
4. Interpret reliability diagrams to diagnose over- and under-coverage
5. Compare parametric distribution heads and select the best-calibrated model

## 1. Choosing the right distribution

Different physical signals have fundamentally different statistical shapes. Picking a distribution that matches that shape is the most important modelling decision in parametric forecasting.

> **Key concept — parametric distributions**
>
> Instead of predicting a single number, a parametric model outputs the *parameters* of a probability distribution — for example, a mean µ and standard deviation σ for the Normal family. The model is trained by maximising the **log-likelihood** of the observed targets under the predicted distribution (equivalently, minimising the **negative log-likelihood**, NLL). This is fundamentally different from pinball/quantile loss, which directly targets specific quantile levels. NLL training uses all the information in the distributional shape, making it more data-efficient when the chosen family is a good match — but poorly calibrated when it is not.
>
> - **Normal**: symmetric, unbounded — the natural default for net load or temperature deltas.
> - **Laplace**: heavier tails than Normal — robust to outlier spikes (electricity prices, residual demand).
> - **Gamma / LogNormal**: strictly positive, right-skewed — ideal for PV generation or aggregate wind.
> - **Beta**: bounded in [0, 1] — suited to capacity factors and state-of-charge signals.

The forecastability profile from NB02 told us `NetLoad(kW)` is approximately symmetric and can go negative — **Normal is the natural starting point**.

In [ ]:
from great_tables import GT, md
import pandas as pd

from twiga.core.plot.gt import twiga_gt

df_dist = pd.DataFrame(
    {
        "Signal characteristic": [
            "Symmetric, can go negative (net load, temp delta)",
            "Heavy-tailed, outlier-prone (price spikes)",
            "Strictly positive, right-skewed (PV, wind)",
            "Bounded [0, 1] (capacity factor, SoC)",
        ],
        "Distribution": ["Normal", "Laplace / StudentT", "LogNormal / Gamma", "Beta"],
        "Config shorthand": [
            'MLPFConfig(distribution="normal")',
            'MLPGAMConfig(distribution="laplace")',
            'MLPFConfig(distribution="lognormal")',
            'MLPGAMConfig(distribution="beta")',
        ],
    }
)

twiga_gt(
    GT(df_dist)
    .tab_header(
        title=md("**Distribution family selector**"),
        subtitle="Match the distribution to the physical shape of your signal",
    )
    .cols_label(**{c: md(f"**{c}**") for c in df_dist.columns})
    .tab_source_note("Twiga Forecast"),
    n_rows=len(df_dist),
)

## 2. Setup

In [ ]:
import warnings

from great_tables import GT
from IPython.display import clear_output
from lets_plot import LetsPlot
import numpy as np
import pandas as pd
from sklearn.preprocessing import RobustScaler, StandardScaler

LetsPlot.setup_html()

from twiga.core.plot import (
    plot_density,
    plot_forecast,
    plot_forecast_grid,
    plot_metrics_bar,
    plot_reliability_diagram,
)
from twiga.core.utils import configure, get_logger

warnings.filterwarnings("ignore")

configure()
log = get_logger("tutorials")

### Load data

In [ ]:
data = pd.read_parquet("../data/MLVS-PT.parquet")
data = data[["timestamp", "NetLoad(kW)", "Ghi", "Temperature"]]
data["timestamp"] = pd.to_datetime(data["timestamp"])
data = data.drop_duplicates(subset="timestamp").reset_index(drop=True)

log.info("Shape: %s", data.shape)
GT(data.head())

### Train / val / test splits

We use the same fixed temporal split as all other tutorials.

In [ ]:
from great_tables import GT, md
import pandas as pd

from twiga.core.plot.gt import twiga_gt

df_splits = pd.DataFrame(
    {
        "Split": ["train", "val", "test"],
        "Period": ["before 2021-01-01", "2021-01-01 – 2021-06-30", "2021-07-01 onwards"],
        "Role": ["Model training", "Early stopping", "Final evaluation"],
    }
)

twiga_gt(
    GT(df_splits)
    .tab_header(title=md("**Data splits**"), subtitle="Fixed temporal partition")
    .cols_label(**{c: md(f"**{c}**") for c in df_splits.columns})
    .tab_source_note("Twiga Forecast"),
    n_rows=len(df_splits),
)

In [ ]:
train_df = data[data["timestamp"] < "2021-01-01"].reset_index(drop=True)
val_df = data[(data["timestamp"] >= "2021-01-01") & (data["timestamp"] < "2021-07-01")].reset_index(drop=True)
test_df = data[data["timestamp"] >= "2021-07-01"].reset_index(drop=True)

log.info(
    f"train : {train_df.shape[0]:,} rows  ({train_df['timestamp'].min().date()} → {train_df['timestamp'].max().date()})"
)
log.info(f"val   : {val_df.shape[0]:,} rows  ({val_df['timestamp'].min().date()} → {val_df['timestamp'].max().date()})")
log.info(
    f"test  : {test_df.shape[0]:,} rows  ({test_df['timestamp'].min().date()} → {test_df['timestamp'].max().date()})"
)

### Data and training configs

In [ ]:
from twiga.core.config import ConformalConfig, DataPipelineConfig, ForecasterConfig

data_config = DataPipelineConfig(
    target_feature="NetLoad(kW)",
    period="30min",
    latitude=32.371666,
    longitude=-16.274998,
    calendar_features=["hour", "day_night"],
    exogenous_features=["Ghi"],
    forecast_horizon=48,
    lookback_window_size=96,
    input_scaler=StandardScaler(),
    target_scaler=RobustScaler(),
)

train_config = ForecasterConfig(split_freq="months", train_size=3, test_size=1)
conformal_config = ConformalConfig(method="residual", alpha=0.1)

data_config

## 3. Gaussian CatBoost

Gaussian CatBoost uses CatBoost's built-in `RMSEWithUncertainty` loss to jointly learn the conditional mean μ and aleatoric uncertainty σ via the negative log-likelihood of a Normal distribution in a **single pass**. Post-training, `predict_with_uncertainty` decomposes total uncertainty into epistemic (knowledge) and aleatoric (data) components via virtual ensemble snapshots.


### 3a. Gaussian CatBoost


In [ ]:
from twiga import TwigaForecaster
from twiga.models.ml import GAUSSCATBOOSTConfig

gauss_cat_config = GAUSSCATBOOSTConfig()
forecaster_gauss_cat = TwigaForecaster(
    data_params=data_config,
    model_params=[gauss_cat_config],
)
forecaster_gauss_cat.fit(train_df)

In [ ]:
pred_gauss_cat, metric_gauss_cat = forecaster_gauss_cat.evaluate_parametric_forecast(test_df=test_df)
clear_output()

log.info("Gaussian CatBoost parametric metrics (mean over days):")
log.info("\n%s", metric_gauss_cat.mean(numeric_only=True).to_string())

### Gaussian CatBoost — interval metrics


In [ ]:
from twiga.core.metrics import get_interval_metrics

gauss_model_results = {
    "Gauss CatBoost": pred_gauss_cat,
}

rows = []
for name, df in gauss_model_results.items():
    m = get_interval_metrics(df)
    m["Model"] = name
    rows.append(m)

interval_df = pd.concat(rows, ignore_index=True)
log.info("Gaussian CatBoost interval metrics:\n%s", interval_df.to_string(index=False))

## 4. The parametric head interface

Every distribution in Twiga is an `nn.Module` that wraps a lightweight linear projection on top of the backbone's latent vector.  They all share the same three-method contract:

- `forward(z)` → distribution parameters as tensors
- `get_distribution(*params)` → a `torch.distributions` object
- `get_log_likelihood(*params, targets)` → negative log-likelihood scalar (the training loss)

The `DISTRIBUTIONS` registry maps string names to classes, and `build_distribution` instantiates them by name.

In [ ]:
import torch

from twiga.distributions.nn import DISTRIBUTIONS, build_distribution

log.info("Available distributions: %s", list(DISTRIBUTIONS.keys()))

# Peek at one head
head = build_distribution("normal", num_target_output=1, hidden_size=64, forecast_horizon=48)
z = torch.randn(4, 64)  # batch of 4 samples
mu, sigma = head(z)
log.info("mu shape   : %s", mu.shape)  # (4, 48, 1)
log.info("sigma shape: %s", sigma.shape)

dist = head.get_distribution(mu, sigma)
samples = dist.sample()
log.info("sample shape: %s", samples.shape)

**Shape convention** — all parametric heads output tensors of shape `(B, forecast_horizon, num_target_output)`, where `B` is the batch size.  This matches the target tensor shape used throughout the training loop, so no reshaping is needed before computing the NLL loss.

## 5. Normal distribution — NetLoad (MLPF backbone)

The Normal head predicts a mean `mu` and a standard deviation `sigma` for every horizon step.  The 90 % prediction interval is `[mu − 1.645σ, mu + 1.645σ]`.  Because `NetLoad(kW)` is approximately symmetric and can go negative, Normal is the textbook choice.

In [ ]:
from twiga import TwigaForecaster
from twiga.models.nn import MLPFConfig

normal_config = MLPFConfig(distribution="normal", max_epochs=5, rich_progress_bar=False)

forecaster_normal = TwigaForecaster(
    data_params=data_config,
    model_params=[normal_config],
    train_params=train_config,
    conformal_params=conformal_config,
)
forecaster_normal.fit(train_df=train_df, val_df=val_df)
clear_output()
forecaster_normal.calibrate(calibrate_df=val_df)

pred_normal_iv, metric_normal_iv = forecaster_normal.evaluate_point_forecast(test_df=test_df)
clear_output()

log.info("Normal — mean interval metrics across folds:")
log.info("\n%s", metric_normal_iv[["mae", "rmse", "corr"]].mean().round(3).to_string())

### Interval forecast

Parametric models predict both location (mean) and scale (uncertainty). A split conformal calibration step uses these predictions on the validation set to guarantee coverage on the test set.

In [ ]:
pred_normal, metric_normal = forecaster_normal.evaluate_parametric_forecast(test_df=test_df)
clear_output()

log.info("Normal — parametric metrics (mean over days):")
log.info("\n%s", metric_normal[["nll", "mae", "rmse"]].mean().round(4).to_string())
GT(pred_normal.head())

### Quick visual — first 3 days

In [ ]:
p = plot_forecast(
    pred_normal_iv.iloc[: 3 * 48],
    actual_col="Actual",
    forecast_col="forecast",
    lower_col="lower",
    upper_col="upper",
    title="MLPFNormal — point forecast + 90% prediction interval (first 3 days of test)",
    y_label="Net Load (kW)",
    x_label="Step (30 min)",
)
p

## 6. Laplace — heavier tails

Laplace has heavier tails than Normal — it assigns more probability to extreme events.  For signals with frequent, sharp spikes (spot electricity prices, residual net load during demand response events), Laplace can produce better-calibrated intervals than Normal while using the same MLPGAM backbone's additive structure.

In [ ]:
from twiga.models.nn import MLPGAMConfig

laplace_config = MLPGAMConfig(distribution="laplace", max_epochs=5, rich_progress_bar=False)

forecaster_laplace = TwigaForecaster(
    data_params=data_config,
    model_params=[laplace_config],
    train_params=train_config,
    conformal_params=conformal_config,
)
forecaster_laplace.fit(train_df=train_df, val_df=val_df)
clear_output()
forecaster_laplace.calibrate(calibrate_df=val_df)

pred_laplace_iv, metric_laplace_iv = forecaster_laplace.evaluate_point_forecast(test_df=test_df)
clear_output()

log.info("Laplace — mean metrics across folds:")
log.info("\n%s", metric_laplace_iv[["mae", "rmse", "corr"]].mean().round(3).to_string())

In [ ]:
pred_laplace, metric_laplace = forecaster_laplace.evaluate_parametric_forecast(test_df=test_df)
clear_output()

log.info("Laplace — parametric metrics (mean over days):")
log.info("\n%s", metric_laplace[["nll", "mae", "rmse"]].mean().round(4).to_string())

### Normal vs Laplace tail comparison

The plot below shows the probability density of each distribution when both have the same mean (0) and the same standard deviation.  Laplace's sharper peak and heavier tails mean it is more robust to outliers during training (equivalent to minimising MAE rather than MSE in the point-forecast limit).

In [ ]:
x = np.linspace(-5, 5, 500)
sigma = 1.0
b = sigma / np.sqrt(2)  # Laplace scale for equal variance

normal_pdf = (1 / (sigma * np.sqrt(2 * np.pi))) * np.exp(-0.5 * (x / sigma) ** 2)
laplace_pdf = (1 / (2 * b)) * np.exp(-np.abs(x) / b)

density_df = pd.DataFrame(
    {
        "value": np.concatenate([x, x]),
        "density": np.concatenate([normal_pdf, laplace_pdf]),
        "distribution": ["Normal (sigma=1)"] * len(x) + ["Laplace (equal variance)"] * len(x),
    }
)

p = plot_density(
    density_df,
    x_col="value",
    color_col="distribution",
    title="Normal vs Laplace — same variance, heavier tails",
    x_label="x",
)
p

## 7. Gamma — strictly positive targets

> **Note**: Gamma requires strictly positive targets. `NetLoad(kW)` can be slightly negative (net export), so we clip negative values for demonstration only. For PV generation (always ≥ 0) or aggregate load with a hard floor, Gamma / LogNormal are the right choice.

The Gamma distribution is parameterised by `concentration` (α) and `rate` (β), with mean α/β and variance α/β².  Unlike Normal and Laplace, it has no `out_activation_function` parameter — both shape parameters are constrained internally via softplus.

In [ ]:
# Clip negative values for demonstration
data_pos = data.copy()
data_pos["NetLoad(kW)"] = data_pos["NetLoad(kW)"].clip(lower=0.01)

train_pos = data_pos[data_pos["timestamp"] < "2021-01-01"].reset_index(drop=True)
val_pos = data_pos[(data_pos["timestamp"] >= "2021-01-01") & (data_pos["timestamp"] < "2021-07-01")].reset_index(
    drop=True
)
test_pos = data_pos[data_pos["timestamp"] >= "2021-07-01"].reset_index(drop=True)

log.info(f"Min NetLoad after clip: {data_pos['NetLoad(kW)'].min():.4f}")

In [ ]:
from twiga.models.nn import MLPFConfig

gamma_config = MLPFConfig(distribution="gamma", max_epochs=5, rich_progress_bar=False)

forecaster_gamma = TwigaForecaster(
    data_params=data_config,
    model_params=[gamma_config],
    train_params=train_config,
    conformal_params=conformal_config,
)
forecaster_gamma.fit(train_df=train_pos, val_df=val_pos)
clear_output()
forecaster_gamma.calibrate(calibrate_df=val_pos)

pred_gamma, metric_gamma = forecaster_gamma.evaluate_point_forecast(test_df=test_pos)
clear_output()

log.info("Gamma — mean metrics across folds:")
log.info("\n%s", metric_gamma[["mae", "rmse", "corr"]].mean().round(3).to_string())

## 8. Evaluating distributional quality

Point metrics (MAE, RMSE) only assess the mean forecast. To evaluate the full predicted distribution we need interval-aware metrics. We use the Normal model's outputs here since it was trained on the original (unclipped) data.

> **Key concept — CRPS**
>
> The **Continuous Ranked Probability Score** (CRPS) measures the entire predictive distribution against the observed outcome in a single number. Formally it is the integrated squared difference between the predicted CDF F and the empirical step-function at the observation y:
>
> CRPS(F, y) = ∫ (F(z) − 𝟙[z ≥ y])² dz
>
> It unifies point and interval evaluation: a degenerate (point) forecast recovers MAE, while a perfectly calibrated distribution achieves the minimum possible CRPS. Lower is better. Unlike interval metrics that target a fixed coverage level, CRPS evaluates the full distributional shape, making it the standard metric for comparing parametric models.

> **Key concept — aleatoric vs. epistemic uncertainty**
>
> Parametric heads capture **aleatoric uncertainty** — the irreducible randomness in the signal itself (e.g. weather-driven demand variability). The predicted σ grows where the data is intrinsically noisy regardless of how much more training data you add. **Epistemic uncertainty** — uncertainty due to limited data or model capacity — is not directly represented by a single parametric head; ensemble methods or Bayesian approaches are needed for that. When interpreting prediction intervals here, you are reading aleatoric uncertainty only.

### Interval metrics from `get_interval_metrics`

In [ ]:
from great_tables import GT, md
import pandas as pd

from twiga.core.plot.gt import twiga_gt

df_metrics = pd.DataFrame(
    {
        "Metric": ["`picp`", "`ace`", "`nmpi`", "`winkle-score`", "`cwe`"],
        "Good value": ["≈ 1 − α", "≈ 0", "low", "low", "high"],
        "Meaning": [
            "Empirical coverage — should match nominal level",
            "Coverage error — positive means under-coverage",
            "Normalised interval width — sharper is better at equal coverage",
            "Penalises both excess width and coverage violations jointly",
            "Combined Width-coverage Error (0–1, higher is better)",
        ],
    }
)

twiga_gt(
    GT(df_metrics)
    .tab_header(
        title=md("**Interval evaluation metrics**"),
        subtitle="From `twiga.core.metrics.get_interval_metrics`",
    )
    .cols_label(**{c: md(f"**{c}**") for c in df_metrics.columns})
    .tab_source_note("Twiga Forecast"),
    n_rows=len(df_metrics),
)

In [ ]:
from twiga.core.metrics import get_interval_metrics

# Aggregate all test steps into flat arrays
true_vals = pred_normal_iv["Actual"].values
pred_vals = pred_normal_iv["forecast"].values
lower_vals = pred_normal_iv["lower"].values
upper_vals = pred_normal_iv["upper"].values

interval_metrics = get_interval_metrics(
    pred=pred_vals,
    true=true_vals,
    lower=lower_vals,
    upper=upper_vals,
    alpha=0.1,
)
log.info("MLPFNormal — interval metrics (90 % nominal coverage):")
log.info("\n%s", interval_metrics.round(4).to_string(index=False))

### Reliability diagram

A reliability diagram checks *calibration*: for every nominal coverage level `1 − α`, the empirical fraction of test points inside the predicted interval should equal `1 − α`. A perfectly calibrated model lies on the diagonal.

**How to read the plot:**
- Points on the diagonal → the model is perfectly calibrated at that level
- Points above the diagonal → over-coverage (intervals are wider than necessary — safe but inefficient)
- Points below the diagonal → under-coverage (intervals are too narrow — the nominal guarantee is not met)
- A systematic upward bow → the distribution family has too heavy tails (e.g. Laplace overestimates spread for a Normal signal)
- A systematic downward bow → the distribution family is too light-tailed, underestimating extreme events

In [ ]:
from scipy.stats import laplace as scipy_laplace, norm as scipy_norm

# Recover distributional parameters from the pre-computed 90% intervals (alpha=0.1)
mu_n = pred_normal_iv["forecast"].values
sigma_n = (pred_normal_iv["upper"].values - mu_n) / scipy_norm.ppf(0.95)
actual = pred_normal_iv["Actual"].values

mu_l = pred_laplace_iv["forecast"].values
b_l = (pred_laplace_iv["upper"].values - mu_l) / (-np.log(0.05))

alphas_rd = [0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.4, 0.5]
cov_normal = []
cov_laplace = []

for a in alphas_rd:
    z = scipy_norm.ppf(1 - a / 2)
    cov_normal.append(((actual >= mu_n - z * sigma_n) & (actual <= mu_n + z * sigma_n)).mean())
    b_ppf = -np.log(a / 2)
    cov_laplace.append(((actual >= mu_l - b_ppf * b_l) & (actual <= mu_l + b_ppf * b_l)).mean())

nominal_levels = [1 - a for a in alphas_rd]
empirical_data = pd.DataFrame(
    {
        "nominal": nominal_levels * 2,
        "empirical": cov_normal + cov_laplace,
        "group": ["Normal (MLPFNormal)"] * len(nominal_levels) + ["Laplace (MLPGAMLaplace)"] * len(nominal_levels),
    }
)

p = plot_reliability_diagram(
    nominal=nominal_levels,
    empirical=cov_normal,
    group_col="group",
    groups=["Normal (MLPFNormal)", "Laplace (MLPGAMLaplace)"],
    title="Reliability diagram",
)
p

## 9. Comparison — Normal vs Laplace vs point

### CRPS — the unified probabilistic metric

**CRPS** (Continuous Ranked Probability Score) is the gold standard for evaluating probabilistic forecasts because it rewards *both* accuracy *and* sharpness simultaneously:

$$\text{CRPS}(F, y) = \int_{-\infty}^{\infty} \bigl(F(x) - \mathbf{1}\{x \geq y\}\bigr)^2 dx$$

where `F` is the predicted CDF and `y` is the observed value.

- **Lower CRPS = better** (it is a negatively oriented score).
- For a deterministic forecast, CRPS degenerates to MAE.
- A distribution that is perfectly centred but too wide will have a higher CRPS than a sharper, well-centred one.

**Interpreting CRPS results:**
- If Normal CRPS < Laplace CRPS: the signal behaves like a Gaussian — Normal is the right family.
- If Laplace CRPS < Normal CRPS: outliers and spikes dominate — Laplace better captures the tail behaviour.
- If point MAE ≈ best CRPS: the uncertainty estimates are not adding value; consider whether the signal is genuinely forecastable.
- CRPS differences of < 1 % are rarely practically significant; focus on large relative gaps.

> **Note**: CRPS requires sampling from the predicted distribution. The cells below compute an empirical CRPS approximation from the predicted `mu` and `sigma` for Normal and the equivalent for Laplace.

In [ ]:
# Empirical CRPS approximation via quantile scores
# CRPS = 2 * mean over quantiles of the pinball loss
def pinball_loss(y_true, y_pred, q):
    """Quantile / pinball loss at level q."""
    e = y_true - y_pred
    return np.where(e >= 0, q * e, (q - 1) * e).mean()


def crps_approx(dist_fn, true_vals, n_quantiles=99):
    """Approximate CRPS by integrating pinball losses over a quantile grid."""
    qs = np.linspace(0.01, 0.99, n_quantiles)
    scores = [pinball_loss(true_vals, dist_fn(q), q) for q in qs]
    return 2 * np.mean(scores)


# Normal: quantile function is mu + sigma * z_q
from scipy.stats import norm as scipy_norm

mu_n = pred_normal_iv["forecast"].values
# Recover sigma from the symmetric interval at alpha=0.1 (upper = mu + z_{0.95} * sigma)
sigma_n = (pred_normal_iv["upper"].values - pred_normal_iv["forecast"].values) / scipy_norm.ppf(0.95)

mu_l = pred_laplace_iv["forecast"].values
# Recover Laplace scale b from the symmetric interval at alpha=0.1
# upper = mu + b * (-log(alpha/2)) = mu + b * (-log(0.05))
b_l = (pred_laplace_iv["upper"].values - pred_laplace_iv["forecast"].values) / (-np.log(0.05))

y_true = pred_normal_iv["Actual"].values

crps_normal = crps_approx(lambda q: mu_n + sigma_n * scipy_norm.ppf(q), y_true)
crps_laplace = crps_approx(lambda q: mu_l + b_l * scipy_laplace.ppf(q), y_true)
crps_point = np.abs(y_true - mu_n).mean()  # CRPS = MAE for deterministic forecast

log.info(f"CRPS (Normal)  : {crps_normal:.4f}")
log.info(f"CRPS (Laplace) : {crps_laplace:.4f}")
log.info(f"MAE  (Normal)  : {crps_point:.4f}  ← CRPS degenerates to MAE for point forecast")

In [ ]:
# Summary comparison table
summary = pd.DataFrame(
    {
        "Model": ["MLPFNormal", "MLPGAMLaplace"],
        "Backbone": ["MLPF", "MLPGAM"],
        "Distribution": ["Normal", "Laplace"],
        "MAE": [
            round(float(metric_normal["mae"].mean()), 3),  # noqa: F821
            round(float(metric_laplace["mae"].mean()), 3),
        ],
        "RMSE": [
            round(float(metric_normal["rmse"].mean()), 3),  # noqa: F821
            round(float(metric_laplace["rmse"].mean()), 3),
        ],
        "CRPS (approx)": [round(crps_normal, 4), round(crps_laplace, 4)],
        "PICP (90%)": [
            round(float(metric_normal_iv["picp"].mean()), 3),
            round(float(metric_laplace_iv["picp"].mean()), 3),
        ],
    }
)

log.info("\n%s", summary.to_string(index=False))

### Interval width comparison — first 48 steps

In [ ]:
n_plot = 48  # one day

normal_slice = pred_normal_iv.iloc[:n_plot].copy()
normal_slice["Model"] = "Normal"
laplace_slice = pred_laplace_iv.iloc[:n_plot].copy()
laplace_slice["Model"] = "Laplace"

combined_iv = pd.concat([normal_slice, laplace_slice], ignore_index=True)

p = plot_forecast_grid(
    combined_iv,
    actual_col="Actual",
    forecast_col="forecast",
    model_col="Model",
    n_samples_per_model=n_plot,
    title="Normal vs Laplace — first 48 steps of test set",
    y_label="Net Load (kW)",
)
p

## Residual Distribution Diagnostics

`plot_kde` and `plot_cdf` let you inspect the shape of the forecast residuals —
a quick sanity check that errors are roughly symmetric and unimodal before
committing to a Gaussian distributional assumption.

In [ ]:
from twiga.core.plot import plot_cdf, plot_kde

residuals_df = pd.DataFrame(
    {
        "residual": pred_normal_iv["Actual"].values - pred_normal_iv["forecast"].values,
    }
)

p_kde = plot_kde(
    residuals_df,
    x_col="residual",
    title="Residual Density — MLPFNormal",
    x_label="Residual (kW)",
)
p_kde

In [ ]:
p_cdf = plot_cdf(
    residuals_df,
    x_col="residual",
    title="Residual Cumulative Distribution — MLPFNormal",
    x_label="Residual (kW)",
)
p_cdf

## Forecast Error Distribution by Step

`plot_distribution` groups by an x-axis column and renders the per-group
mean ± 1σ ribbon — useful for diagnosing whether errors are uniformly
distributed across the forecast horizon or worsen at specific steps.

In [ ]:
from twiga.core.plot import plot_distribution

dist_df = pred_normal_iv.copy()
dist_df["step"] = np.arange(len(dist_df)) % data_config.forecast_horizon
dist_df["residual"] = dist_df["Actual"] - dist_df["forecast"]

p_dist = plot_distribution(
    dist_df,
    x_col="step",
    y_col="residual",
    title="Residual Distribution by Forecast Step — MLPFNormal",
    x_label="Step (30 min)",
    y_label="Residual (kW)",
)
p_dist

## PIT Histogram

The Probability Integral Transform (PIT) histogram is the gold-standard
calibration diagnostic for parametric forecasts (Gneiting et al., 2007).
A flat histogram indicates perfect calibration; a hump shape means the model
is overconfident (intervals too narrow); a U-shape means it is over-dispersed.

In [ ]:
from twiga.core.plot import plot_pit_histogram

# Recover sigma from the pre-computed 90 % interval: upper = mu + z_0.95 * sigma
mu_pit = pred_normal_iv["forecast"].values
sigma_pit = (pred_normal_iv["upper"].values - mu_pit) / scipy_norm.ppf(0.95)
y_pit = pred_normal_iv["Actual"].values

pit_values = scipy_norm.cdf(y_pit, loc=mu_pit, scale=sigma_pit)

p_pit = plot_pit_histogram(
    pit_values,
    n_bins=20,
    title="PIT Histogram — MLPFNormal",
)
p_pit

## Reliability Diagram

Plots empirical coverage against nominal coverage levels.
Points on the diagonal = perfectly calibrated.
Points above = conservative (intervals too wide).
Points below = anti-conservative (intervals too narrow).

In [ ]:
from twiga.core.plot import plot_reliability_diagram

nominal_levels = np.linspace(0.10, 0.95, 18)

mu_rd = pred_normal_iv["forecast"].values
sigma_rd = (pred_normal_iv["upper"].values - mu_rd) / scipy_norm.ppf(0.95)
y_rd = pred_normal_iv["Actual"].values

empirical_coverage = np.array(
    [
        (
            (y_rd >= mu_rd - scipy_norm.ppf(1 - (1 - lvl) / 2) * sigma_rd)
            & (y_rd <= mu_rd + scipy_norm.ppf(1 - (1 - lvl) / 2) * sigma_rd)
        ).mean()
        for lvl in nominal_levels
    ]
)

p_rel = plot_reliability_diagram(
    nominal=nominal_levels,
    empirical=empirical_coverage,
    title="Reliability Diagram — MLPFNormal",
)
p_rel

## Wrapping up

**What you did**
- [x] Chose distribution families based on signal characteristics (Normal, Laplace, Gamma)
- [x] Configured parametric distribution heads on MLPF and MLPGAM backbone architectures
- [x] Trained models end-to-end with NLL loss and extracted predictive mean and credible intervals
- [x] Compared Normal vs. Laplace tails with density plots and residual diagnostics
- [x] Evaluated distributional quality using CRPS, PIT histograms, and reliability diagrams

**Key takeaways**
1. Parametric models output distribution parameters (µ, σ) — not point predictions — and are trained by minimising NLL rather than a point loss.
2. The choice of distribution family matters: Normal for symmetric signals, Laplace for heavy-tailed ones, Gamma/LogNormal for strictly positive targets.
3. CRPS is the gold-standard metric for comparing parametric models — it evaluates the full distributional shape in a single score.
4. A well-calibrated model tracks the diagonal of the reliability diagram; systematic deviations reveal distributional mismatch.
5. Parametric heads capture aleatoric (irreducible) uncertainty — epistemic uncertainty requires ensembles or Bayesian approaches.

---

## What's next?

**NB09 — Conformal Prediction** (`09-conformal-prediction.ipynb`)

Learn how to wrap any fitted Twiga model with a coverage-guaranteed conformal calibration step. Conformal prediction requires no distributional assumption and works with any base model — including the parametric models you trained here.

In [ ]:
# ruff: noqa: E501, E701, E702
from IPython.display import HTML

_TEAL = "#107591"
_TEAL_MID = "#069fac"
_TEAL_LIGHT = "#e8f5f8"
_TEAL_BEST = "#d0ecf1"
_TEXT_DARK = "#2d3748"
_TEXT_MUTED = "#718096"
_WHITE = "#ffffff"

steps = [
    {
        "num": "07",
        "title": "Neural Networks",
        "desc": "MLPF · N-HiTS · Lightning training · sequence embeddings",
        "tags": ["neural network", "pytorch"],
        "active": False,
    },
    {
        "num": "08",
        "title": "Quantile Regression",
        "desc": "QR-LightGBM · FPQR — calibrated prediction intervals",
        "tags": ["quantile", "pinball loss"],
        "active": False,
    },
    {
        "num": "09",
        "title": "Parametric Distributions",
        "desc": "Normal · Laplace · Gamma heads — NLL training · CRPS evaluation",
        "tags": ["parametric", "NLL", "CRPS"],
        "active": True,
    },
    {
        "num": "10",
        "title": "Conformal Prediction",
        "desc": "Coverage-guaranteed intervals with CQR and CRC wrappers",
        "tags": ["conformal", "CQR", "coverage"],
        "active": False,
    },
    {
        "num": "11",
        "title": "Hyperparameter Tuning",
        "desc": "Optuna-backed HPO · search spaces · resumable SQLite",
        "tags": ["optuna", "HPO", "tuning"],
        "active": False,
    },
]
track_name = "Probabilistic Track"
footer = 'Next: wrap any model with <span style="color:#107591;font-weight:600;">Conformal Prediction</span> (10) for finite-sample coverage guarantees.'


def _b(t, bg, fg):
    return f'<span style="display:inline-block;background:{bg};color:{fg};font-size:10px;font-weight:600;padding:2px 7px;border-radius:10px;margin:2px 2px 0 0;">{t}</span>'


ch = ""
for i, s in enumerate(steps):
    a = s["active"]
    cb = _TEAL if a else _WHITE
    cbo = _TEAL if a else "#d1ecf1"
    nb = _TEAL_MID if a else _TEAL_LIGHT
    nf = _WHITE if a else _TEAL
    tf = _WHITE if a else _TEXT_DARK
    df = "#cce8ef" if a else _TEXT_MUTED
    bb = "#0d5f75" if a else _TEAL_BEST
    bf = "#b8e4ed" if a else _TEAL
    yh = (
        f'<span style="float:right;background:{_TEAL_MID};color:{_WHITE};font-size:10px;font-weight:700;padding:2px 10px;border-radius:12px;">★ you are here</span>'
        if a
        else ""
    )
    bdg = "".join(_b(t, bb, bf) for t in s["tags"])
    ch += f'<div style="background:{cb};border:2px solid {cbo};border-radius:12px;padding:16px 20px;display:flex;align-items:flex-start;gap:16px;box-shadow:{"0 4px 14px rgba(16,117,145,.25)" if a else "0 1px 4px rgba(0,0,0,.06)"};"><div style="min-width:44px;height:44px;background:{nb};color:{nf};border-radius:50%;display:flex;align-items:center;justify-content:center;font-size:15px;font-weight:800;flex-shrink:0;">{s["num"]}</div><div style="flex:1;"><div style="font-size:15px;font-weight:700;color:{tf};margin-bottom:4px;">{s["title"]}{yh}</div><div style="font-size:12.5px;color:{df};margin-bottom:8px;line-height:1.5;">{s["desc"]}</div><div>{bdg}</div></div></div>'
    if i < len(steps) - 1:
        ch += f'<div style="display:flex;justify-content:center;height:32px;"><svg width="24" height="32" viewBox="0 0 24 32" fill="none"><line x1="12" y1="0" x2="12" y2="24" stroke="{_TEAL_MID}" stroke-width="2" stroke-dasharray="4 3"/><polygon points="6,20 18,20 12,30" fill="{_TEAL_MID}"/></svg></div>'

HTML(
    f'<div style="font-family:Inter,\'Segoe UI\',sans-serif;max-width:640px;margin:8px 0;"><div style="background:linear-gradient(135deg,{_TEAL} 0%,{_TEAL_MID} 100%);border-radius:12px 12px 0 0;padding:14px 20px;display:flex;align-items:center;gap:10px;"><svg width="22" height="22" viewBox="0 0 24 24" fill="none" stroke="{_WHITE}" stroke-width="2"><path d="M12 2L2 7l10 5 10-5-10-5z"/><path d="M2 17l10 5 10-5"/><path d="M2 12l10 5 10-5"/></svg><span style="color:{_WHITE};font-size:14px;font-weight:700;">Twiga Learning Path — {track_name}</span></div><div style="border:2px solid {_TEAL_LIGHT};border-top:none;border-radius:0 0 12px 12px;padding:20px 20px 16px;background:#f9fdfe;display:flex;flex-direction:column;">{ch}<div style="margin-top:16px;font-size:11.5px;color:{_TEXT_MUTED};text-align:center;border-top:1px solid {_TEAL_LIGHT};padding-top:12px;">{footer}</div></div></div>'
)